In [1]:
import pandas as pd
import sqlite3
import os

print("Esame aplanke:", os.getcwd())
print("\nFailai šiame aplanke:")
for f in os.listdir():
    print(" -", f)

Esame aplanke: C:\Users\gabri\Documents\nq-monte-carlo

Failai šiame aplanke:
 - .ipynb_checkpoints
 - 01_data_exploration.ipynb
 - 02_sql_setup.ipynb
 - nq_15min.csv
 - nyc_weather.csv


In [2]:
conn = sqlite3.connect('nq_project.db')
print("Bazė sukurta!")
print("\nFailai dabar aplanke:")
for f in os.listdir():
    print(" -", f)

Bazė sukurta!

Failai dabar aplanke:
 - .ipynb_checkpoints
 - 01_data_exploration.ipynb
 - 02_sql_setup.ipynb
 - nq_15min.csv
 - nq_project.db
 - nyc_weather.csv


In [3]:
# NQ duomenys
df = pd.read_csv('nq_15min.csv', sep='\t')
df['DateTime'] = pd.to_datetime(df['DateTime'])
df = df.sort_values('DateTime').reset_index(drop=True)
df = df.drop(columns=['Volume'])

# Orų duomenys
weather = pd.read_csv('nyc_weather.csv')
weather['DATE'] = pd.to_datetime(weather['DATE'])
weather = weather.drop(columns=['STATION', 'NAME'])
weather = weather.rename(columns={'DATE': 'Date', 'PRCP': 'Precipitation'})

print(f"NQ duomenys: {len(df)} eilučių")
print(f"Orų duomenys: {len(weather)} eilučių")
print("\nViskas paruošta!")

NQ duomenys: 206703 eilučių
Orų duomenys: 3256 eilučių

Viskas paruošta!


In [4]:
weather.to_sql('weather', conn, if_exists='replace', index=False)
print("Orų duomenys įkelti į lentelę 'weather'")

# Patikrinam, kad duomenys tikrai pateko į bazę
test_query = pd.read_sql('SELECT COUNT(*) AS row_count FROM weather', conn)
print(f"Eilučių lentelėje 'weather': {test_query['row_count'][0]}")

Orų duomenys įkelti į lentelę 'weather'
Eilučių lentelėje 'weather': 3256


In [5]:
df.to_sql('market_data', conn, if_exists='replace', index=False)
print("NQ duomenys įkelti į lentelę 'market_data'")

test_query = pd.read_sql('SELECT COUNT(*) AS row_count FROM market_data', conn)
print(f"Eilučių lentelėje 'market_data': {test_query['row_count'][0]}")

NQ duomenys įkelti į lentelę 'market_data'
Eilučių lentelėje 'market_data': 206703


In [6]:
# Filtruojam tik 09:30 ir 16:30 žvakes
df['Time'] = df['DateTime'].dt.time
target_times = [pd.to_datetime('09:30').time(), pd.to_datetime('16:30').time()]
daily = df[df['Time'].isin(target_times)].copy()
daily['Date'] = daily['DateTime'].dt.date

# Pivot į "vieną eilutę per dieną"
daily_wide = daily.pivot(index='Date', columns='Time', values='Close')
daily_wide.columns = ['ORB_Close', 'EOD_Close']
daily_wide = daily_wide.reset_index()
daily_wide['Day_Move'] = daily_wide['EOD_Close'] - daily_wide['ORB_Close']
daily_wide['Date'] = pd.to_datetime(daily_wide['Date'])

# Merge su orais
merged = daily_wide.merge(weather, on='Date', how='inner')

# Pridedam strategijos stulpelius
merged['Rainy'] = merged['Precipitation'] > 0
merged['Direction'] = merged['Rainy'].apply(lambda x: 'Short' if x else 'Long')
merged['Weather_PnL'] = merged.apply(
    lambda row: row['Day_Move'] if row['Direction'] == 'Long' else -row['Day_Move'],
    axis=1
)

# Pašalinam NaN (early closes)
merged_clean = merged.dropna(subset=['Day_Move']).reset_index(drop=True)

print(f"daily_results paruošta: {len(merged_clean)} eilučių")
merged_clean.head()

daily_results paruošta: 2266 eilučių


,Date,ORB_Close,EOD_Close,Day_Move,Precipitation,Rainy,Direction,Weather_PnL
0,2016-11-16,4764.2,4777.2,13.0,0.0,False,Long,13.0
1,2016-11-17,4796.7,4791.8,-4.9,0.0,False,Long,-4.9
2,2016-11-18,4834.2,4831.9,-2.3,0.0,False,Long,-2.3
3,2016-11-21,4821.0,4839.0,18.0,0.0,False,Long,18.0
4,2016-11-22,4880.0,4878.3,-1.7,0.0,False,Long,-1.7


In [7]:
merged_clean.to_sql('daily_results', conn, if_exists='replace', index=False)
print("Strategijos rezultatai įkelti į lentelę 'daily_results'")

test_query = pd.read_sql('SELECT COUNT(*) AS row_count FROM daily_results', conn)
print(f"Eilučių lentelėje 'daily_results': {test_query['row_count'][0]}")

Strategijos rezultatai įkelti į lentelę 'daily_results'
Eilučių lentelėje 'daily_results': 2266


In [8]:
import numpy as np

# Perskaičiuojam Markovo matricą (reikia jos simuliacijai)
merged_clean['Prev_Rainy'] = merged_clean['Rainy'].shift(1)
transitions = merged_clean.dropna(subset=['Prev_Rainy'])
transition_matrix = pd.crosstab(
    transitions['Prev_Rainy'], 
    transitions['Rainy'], 
    normalize='index'
)

p_rain_after_dry = transition_matrix.loc[False, True]
p_rain_after_rain = transition_matrix.loc[True, True]

# Monte Carlo parametrai
n_simulations = 10000
n_days = len(merged_clean)
real_moves = merged_clean['Day_Move'].values

# Vieta rezultatams
sim_results = np.zeros(n_simulations)

np.random.seed(42)

for sim in range(n_simulations):
    synthetic_rain = np.zeros(n_days, dtype=bool)
    synthetic_rain[0] = np.random.random() < 0.361
    
    for day in range(1, n_days):
        if synthetic_rain[day-1]:
            synthetic_rain[day] = np.random.random() < p_rain_after_rain
        else:
            synthetic_rain[day] = np.random.random() < p_rain_after_dry
    
    sim_pnl = np.where(synthetic_rain, -real_moves, real_moves).sum()
    sim_results[sim] = sim_pnl

print(f"Monte Carlo baigta: {n_simulations} simuliacijų")
print(f"Vidutinis PnL: {sim_results.mean():.1f}")

Monte Carlo baigta: 10000 simuliacijų
Vidutinis PnL: 282.9


In [9]:
# Paverčiam numpy masyvą į DataFrame'ą su simulation ID
mc_df = pd.DataFrame({
    'simulation_id': range(1, n_simulations + 1),
    'total_pnl': sim_results
})

# Įkeliam į bazę
mc_df.to_sql('mc_simulations', conn, if_exists='replace', index=False)
print("Monte Carlo rezultatai įkelti į lentelę 'mc_simulations'")

# Patikrinam
test_query = pd.read_sql('SELECT COUNT(*) AS row_count FROM mc_simulations', conn)
print(f"Eilučių lentelėje 'mc_simulations': {test_query['row_count'][0]}")

mc_df.head()

Monte Carlo rezultatai įkelti į lentelę 'mc_simulations'
Eilučių lentelėje 'mc_simulations': 10000


,simulation_id,total_pnl
0,1,2327.2
1,2,-10000.8
2,3,-4850.6
3,4,-9120.6
4,5,-4815.4


In [10]:
conn.commit()
conn.close()
print("Bazė išsaugota ir uždaryta.")

Bazė išsaugota ir uždaryta.


In [11]:
# Atnaujinam ryšį (prisimeni, close() uždarėme anksčiau)
conn = sqlite3.connect('nq_project.db')

# Perskaitom, suapvalinam, perrašom atgal
daily_results = pd.read_sql('SELECT * FROM daily_results', conn)
daily_results['ORB_Close'] = daily_results['ORB_Close'].round(2)
daily_results['EOD_Close'] = daily_results['EOD_Close'].round(2)
daily_results['Day_Move'] = daily_results['Day_Move'].round(2)
daily_results['Weather_PnL'] = daily_results['Weather_PnL'].round(2)
daily_results['Precipitation'] = daily_results['Precipitation'].round(2)

daily_results.to_sql('daily_results', conn, if_exists='replace', index=False)

mc_df = pd.read_sql('SELECT * FROM mc_simulations', conn)
mc_df['total_pnl'] = mc_df['total_pnl'].round(2)
mc_df.to_sql('mc_simulations', conn, if_exists='replace', index=False)

conn.commit()
conn.close()

print("Skaičiai suapvalinti, bazė atnaujinta")

Skaičiai suapvalinti, bazė atnaujinta


In [12]:
# Sukuriam 'exports' aplanką CSV failams
os.makedirs('exports', exist_ok=True)

# Atidarom ryšį
conn = sqlite3.connect('nq_project.db')

# Eksportuojam kiekvieną lentelę į CSV
tables = ['weather', 'daily_results', 'mc_simulations']

for table in tables:
    df_export = pd.read_sql(f'SELECT * FROM {table}', conn)
    df_export.to_csv(f'exports/{table}.csv', index=False)
    print(f"Eksportuota: exports/{table}.csv ({len(df_export)} eilučių)")

conn.close()
print("\nVisos lentelės eksportuotos.")

Eksportuota: exports/weather.csv (3256 eilučių)
Eksportuota: exports/daily_results.csv (2266 eilučių)
Eksportuota: exports/mc_simulations.csv (10000 eilučių)

Visos lentelės eksportuotos.
